In [1]:
from pathlib import Path
import gzip
import scipy.io
import anndata as ad
import pandas as pd
import scanpy as sp
import numpy as np
import scipy as sc
import requests
from io import StringIO
from collections import defaultdict
#Test one vs Rest an use The wilcoxon ranksum test in scipy 
from scipy.stats import ranksums

In [2]:
metadata = pd.read_csv(
    "GSE164378_sc.meta.data_3P.csv.gz",
    index_col=0,
    compression="gzip"
)

print(metadata.head())
print(metadata.shape)

raw_dir = Path("GSE164378_RAW")  # ggf. anpassen

matrix_path = raw_dir / "GSM5008737_RNA_3P-matrix.mtx.gz"
barcodes_path = raw_dir / "GSM5008737_RNA_3P-barcodes.tsv.gz"
features_path = raw_dir / "GSM5008737_RNA_3P-features.tsv.gz"

# load the 3P matrix and transpose it to have cells x genes and convert iot to a sparsematrix
X = scipy.io.mmread(matrix_path).T.tocsr()

# Load the cellbarcodes
with gzip.open(barcodes_path, "rt") as f:
    barcodes = [line.strip() for line in f]

# load the genes
features = pd.read_csv(
    features_path,
    sep="\t",
    header=None,
    compression="gzip"
)

gene_names = features[0].values

# Create the AnnData Object
adata_rna = ad.AnnData(
    X=X,
    obs=pd.DataFrame(index=barcodes),
    var=pd.DataFrame(index=gene_names)
)

#Repeat with the ADT Data
matrix_path = raw_dir / "GSM5008738_ADT_3P-matrix.mtx.gz"
barcodes_path = raw_dir / "GSM5008738_ADT_3P-barcodes.tsv.gz"
features_path = raw_dir / "GSM5008738_ADT_3P-features.tsv.gz"

X_adt = scipy.io.mmread(matrix_path).T.tocsr()

with gzip.open(barcodes_path, "rt") as f:
    adt_barcodes = [line.strip() for line in f]

adt_features = pd.read_csv(
    features_path,
    sep="\t",
    header=None,
    compression="gzip"
)

protein_names = adt_features[0].values

adata_adt = ad.AnnData(
    X=X_adt,
    obs=pd.DataFrame(index=adt_barcodes),
    var=pd.DataFrame(index=protein_names)
)

#Merge the meta data into the anndata object
adata_rna.obs = metadata.loc[adata_rna.obs_names].copy()
adata_rna.obs.head()

adata_adt.obs = metadata.loc[adata_adt.obs_names].copy()
adata_adt.obs.head()

                     nCount_ADT  nFeature_ADT  nCount_RNA  nFeature_RNA  \
L1_AAACCCAAGAAACTCA        7535           217       10823          2915   
L1_AAACCCAAGACATACA        6013           209        5864          1617   
L1_AAACCCACAACTGGTT        6620           213        5067          1381   
L1_AAACCCACACGTACTA        3567           202        4786          1890   
L1_AAACCCACAGCATACT        6402           215        6505          1621   

                        orig.ident lane donor  time celltype.l1 celltype.l2  \
L1_AAACCCAAGAAACTCA  SeuratProject   L1    P2     7        Mono   CD14 Mono   
L1_AAACCCAAGACATACA  SeuratProject   L1    P1     7       CD4 T     CD4 TCM   
L1_AAACCCACAACTGGTT  SeuratProject   L1    P4     2       CD8 T   CD8 Naive   
L1_AAACCCACACGTACTA  SeuratProject   L1    P3     7          NK          NK   
L1_AAACCCACAGCATACT  SeuratProject   L1    P4     7       CD8 T   CD8 Naive   

                    celltype.l3 Phase   Batch  
L1_AAACCCAAGAAACTCA   CD14

,nCount_ADT,nFeature_ADT,nCount_RNA,nFeature_RNA,orig.ident,lane,donor,time,celltype.l1,celltype.l2,celltype.l3,Phase,Batch
L1_AAACCCAAGAAACTCA,7535,217,10823,2915,SeuratProject,L1,P2,7,Mono,CD14 Mono,CD14 Mono,G1,Batch1
L1_AAACCCAAGACATACA,6013,209,5864,1617,SeuratProject,L1,P1,7,CD4 T,CD4 TCM,CD4 TCM_1,G1,Batch1
L1_AAACCCACAACTGGTT,6620,213,5067,1381,SeuratProject,L1,P4,2,CD8 T,CD8 Naive,CD8 Naive,S,Batch1
L1_AAACCCACACGTACTA,3567,202,4786,1890,SeuratProject,L1,P3,7,NK,NK,NK_2,G1,Batch1
L1_AAACCCACAGCATACT,6402,215,6505,1621,SeuratProject,L1,P4,7,CD8 T,CD8 Naive,CD8 Naive,G1,Batch1


In [5]:
import requests
from io import StringIO

def search_uniprot_protein_to_gene(protein_names, organism_id=9606, reviewed_only=True):
    all_results = []

    for protein in protein_names:
        query = f'(protein_name:"{protein}" OR gene:{protein}) AND organism_id:{organism_id}'
        
        if reviewed_only:
            query += " AND reviewed:true"

        url = "https://rest.uniprot.org/uniprotkb/search"

        params = {
            "query": query,
            "fields": "accession,id,protein_name,gene_primary,organism_name,reviewed",
            "format": "tsv",
            "size": 10
        }

        r = requests.get(url, params=params)
        r.raise_for_status()

        if r.text.strip():
            df = pd.read_csv(StringIO(r.text), sep="\t")
            df.insert(0, "Surface_Protein", protein)
            all_results.append(df)
        else:
            all_results.append(pd.DataFrame({
                "Surface_Protein": [protein],
                "Entry": [None],
                "Gene Names": [None]
            }))

    return pd.concat(all_results, ignore_index=True)



In [27]:
def load_groundtruth_as_dict(path_to_groundtruth_csv):
    groundtruth = pd.read_csv(path_to_groundtruth_csv)    
    protein_to_genes = defaultdict(list)

    for _, row in groundtruth.iterrows():
        protein = row["Surface_Protein"]
        genes = row["Gene Names (primary)"]

        if pd.isna(genes):
            continue

        protein_to_genes[protein].extend(genes.split())

    # Duplikate entfernen, Reihenfolge behalten
    protein_to_genes = {
        protein: list(dict.fromkeys(genes))
        for protein, genes in protein_to_genes.items()
    }
    return protein_to_genes

In [3]:
gene_names_dataset = adata_rna.var_names.tolist()
gene_names_dataset

['MIR1302-2HG',
 'FAM138A',
 'OR4F5',
 'AL627309.1',
 'AL627309.3',
 'AL627309.2',
 'AL627309.4',
 'AL732372.1',
 'OR4F29',
 'AC114498.1',
 'OR4F16',
 'AL669831.2',
 'AL669831.5',
 'FAM87B',
 'LINC00115',
 'FAM41C',
 'AL645608.7',
 'AL645608.3',
 'AL645608.5',
 'AL645608.1',
 'SAMD11',
 'NOC2L',
 'KLHL17',
 'PLEKHN1',
 'PERM1',
 'AL645608.8',
 'HES4',
 'ISG15',
 'AL645608.2',
 'AGRN',
 'AL645608.9',
 'RNF223',
 'C1orf159',
 'LINC01342',
 'AL390719.2',
 'TTLL10-AS1',
 'TTLL10',
 'TNFRSF18',
 'TNFRSF4',
 'SDF4',
 'B3GALT6',
 'C1QTNF12',
 'AL162741.1',
 'UBE2J2',
 'LINC01786',
 'SCNN1D',
 'ACAP3',
 'PUSL1',
 'INTS11',
 'CPTP',
 'TAS1R3',
 'DVL1',
 'MXRA8',
 'AURKAIP1',
 'CCNL2',
 'MRPL20',
 'AL391244.3',
 'ANKRD65',
 'AL391244.2',
 'TMEM88B',
 'LINC01770',
 'VWA1',
 'ATAD3C',
 'ATAD3B',
 'ATAD3A',
 'TMEM240',
 'SSU72',
 'AL645728.1',
 'FNDC10',
 'AL691432.2',
 'MIB2',
 'MMP23B',
 'CDK11B',
 'FO704657.1',
 'SLC35E2B',
 'CDK11A',
 'SLC35E2A',
 'NADK',
 'GNB1',
 'AL109917.1',
 'CALML6',
 'TM

In [6]:
all_adt_df = search_uniprot_protein_to_gene(adata_adt.var_names.tolist())

In [26]:
all_adt_df.to_csv("GENES_all_228_adts.csv", index=False)

In [11]:
gene_adt_set = set(all_adt_df["Gene Names (primary)"])


In [12]:
values_PROT = list(PROTEIN_TO_GENES.values())
PROT_set = set(element for sublist in values_PROT for element in sublist)


In [20]:
whole_rna_gene_set = set(adata_rna.var_names.tolist())


In [21]:
len(whole_rna_gene_set.intersection(PROT_set))

155

In [23]:
print("Number Genes found: ", len(gene_adt_set))
print("Length Intersection: ", len(whole_rna_gene_set.intersection(gene_adt_set)))

Number Genes found:  254
Length Intersection:  244


In [24]:
len(gene_adt_set.intersection(PROT_set))

96

In [28]:
abra = load_groundtruth_as_dict("GENES_all_228_adts.csv")

In [1]:
uniprot_genes_228_adts = {'CD39': ['ENTPD1', 'ENTPD6', 'ENTPD2', 'ENTPD3', 'ENTPD5'],
 'CD107a': ['LAMP1'],
 'CD62P': ['SELP'],
 'CD30': ['TNFRSF8', 'TNFSF8'],
 'CD31': ['PECAM1'],
 'CD34': ['CD34'],
 'CD35': ['CR1'],
 'CD36': ['CD36', 'SCARB1', 'SCARB2'],
 'CD223': ['LAG3'],
 'TIGIT': ['TIGIT'],
 'CD226': ['CD226'],
 'CD178': ['FASLG'],
 'CD319': ['SLAMF7'],
 'CD171': ['L1CAM'],
 'Siglec-8': ['SIGLEC8'],
 'CD340': ['ERBB2'],
 'VEGFR-3': ['FLT4'],
 'CD29': ['ITGB1'],
 'CD62E': ['SELE'],
 'CD22': ['CD22'],
 'CD20': ['MS4A1', 'MS4A7', 'MS4A6A', 'MS4A4A', 'MS4A3', 'MS4A10', 'MS4A5'],
 'CD27': ['CD27', 'CD70', 'SIVA1'],
 'CD25': ['IL2RA'],
 'CD24': ['CD24'],
 'CD146': ['MCAM'],
 'Galectin-9': ['LGALS9C', 'LGALS9', 'LGALS9B'],
 'CD142': ['F3'],
 'CD141': ['THBD'],
 'CD294': ['PTGDR2'],
 'CX3CR1': ['CX3CR1'],
 'CD303': ['CLEC4C'],
 'GP130': ['LRPPRC', 'IL31RA', 'IL6ST', 'TLE5'],
 'CD253': ['TNFSF10'],
 'CD357': ['TNFRSF18'],
 'CD354': ['TREM1'],
 'CLEC12A': ['CLEC12A'],
 'Folate': ['FOLR2',
  'FOLR3',
  'FOLR1',
  'SLC19A1',
  'IZUMO1R',
  'SLC46A1',
  'FOLH1B',
  'FOLH1',
  'SLC19A4P'],
 'CD209': ['CD209', 'CLEC4M'],
 'CD152': ['CTLA4'],
 'CD154': ['CD40LG'],
 'CD155': ['PVR'],
 'Cadherin': ['CDH15',
  'CDH6',
  'CDH3',
  'CDH12',
  'CDH13',
  'CDH16',
  'CDH10',
  'CDH18',
  'CDH11',
  'CDH2'],
 'CD201': ['PROCR'],
 'CD204': ['MSR1'],
 'CD205': ['LY75'],
 'CD206': ['MRC1'],
 'CD207': ['CD207'],
 'CD1d': ['CD1D'],
 'CD284': ['TLR4'],
 'CD1c': ['CD1C'],
 'Podoplanin': ['PDPN'],
 'CD1a': ['CD1A'],
 'CD366': ['HAVCR2'],
 'IgM': ['JCHAIN', 'CD5L', 'FCMR', 'CD79A'],
 'CD49d': ['ITGA4'],
 'LOX-1': ['ALOX15', 'OLR1'],
 'TIM-4': ['TIMD4'],
 'CD98': ['SLC3A2', 'SLC7A5'],
 'CD370': ['CLEC9A'],
 'CD49a': ['ITGA1'],
 'C5L2': ['C5AR2'],
 'CD124': ['IL4R'],
 'CD127': ['IL7R'],
 'CD126': ['IL6R'],
 'CD279': ['PDCD1'],
 'CD278': ['ICOS'],
 'CD123': ['IL3RA'],
 'CD122': ['IL2RB'],
 'CD96': ['CD96'],
 'CD274': ['CD274'],
 'CD95': ['FAS', 'FASLG'],
 'CD271': ['NGFR'],
 'CD270': ['TNFRSF14'],
 'CD90': ['THY1'],
 'CD272': ['BTLA'],
 'CD16': ['FCGR3B', 'FCGR3A'],
 'CD14': ['CD14'],
 'CD13': ['ANPEP'],
 'CD267': ['TNFRSF13B'],
 'CD200': ['CD200', 'CD200R1', 'CD200R1L'],
 'CD18': ['ITGB2'],
 'CD19': ['CD19'],
 'CD194': ['CCR4'],
 'CD70': ['CD70'],
 'CD71': ['TFRC'],
 'CD72': ['CD72'],
 'CD73': ['NT5E'],
 'CD177': ['CD177'],
 'CD301': ['CLEC10A'],
 'CD140a': ['PDGFRA'],
 'CD140b': ['PDGFRB'],
 'CD305': ['LAIR1'],
 'CD304': ['NRP1'],
 'CD2': ['CD2', 'CD2BP2', 'SLAMF7', 'CD2AP', 'PSTPIP1', 'SH3KBP1', 'SLAMF9'],
 'CD309': ['KDR'],
 'CD85g': ['LILRA4'],
 'CD110': ['MPL'],
 'CD8': ['CD8B', 'CD8A', 'CD8B2'],
 'CD9': ['CD9', 'PTGFRN'],
 'HLA-DR': ['ANP32B', 'SET', 'CD74', 'ANP32A'],
 'CD137': ['TNFRSF9'],
 'CD134': ['TNFRSF4'],
 'CD135': ['FLT3'],
 'CD61': ['ITGB3'],
 'CD192': ['CCR2'],
 'CD268': ['TNFRSF13C'],
 'CD269': ['TNFRSF17'],
 'CD81': ['CD81', 'IGSF8'],
 'CD80': ['CD80'],
 'CD83': ['CD83'],
 'CD193': ['CCR3'],
 'TSLPR': ['CRLF2'],
 'CD86': ['CD86'],
 'CCR10': ['CCR10', 'ACKR2'],
 'Notch-1': ['NOTCH1'],
 'Notch-2': ['NOTCH2'],
 'CD337': ['NCR3'],
 'CD79b': ['CD79B'],
 'CD79a': ['CD79A', 'IGBP1'],
 'CD49b': ['ITGA2'],
 'CD64': ['FCGR1A'],
 'CD63': ['CD63'],
 'CD69': ['CD69'],
 'CD68': ['CD68'],
 'CD314': ['KLRK1'],
 'CD186': ['CXCR6'],
 'CD185': ['CXCR5'],
 'CD184': ['CXCR4'],
 'CD103': ['ITGAE'],
 'CD102': ['ICAM2'],
 'CD106': ['VCAM1'],
 'CD105': ['ENG'],
 'CD66b': ['CEACAM8'],
 'CD252': ['TNFSF4'],
 'CD109': ['CD109'],
 'CD158f': ['KIR2DL5B', 'KIR2DL5A'],
 'CD8a': ['CD8A'],
 'CD203c': ['ENPP3'],
 'CD52': ['CD52'],
 'CD195': ['CCR5'],
 'CD196': ['CCR6'],
 'CD54': ['ICAM1'],
 'CD55': ['CD55'],
 'CD99': ['CD99', 'CD99L2'],
 'CD59': ['CD59'],
 'CD93': ['CD93'],
 'CD244': ['CD244'],
 'CD158': ['KIR3DL3',
  'KIR2DL5B',
  'KIR2DS5',
  'KIR2DS1',
  'KIR2DL4',
  'KIR2DL2',
  'KIR3DL2',
  'KIR2DS4',
  'KIR2DS2',
  'KIR3DL1'],
 'CD273': ['PDCD1LG2'],
 'CD243': ['ABCB1'],
 'CD325': ['CDH2'],
 'CD324': ['CDH1'],
 'CD307e': ['FCRL5'],
 'CD172a': ['SIRPA'],
 'CD307d': ['FCRL4'],
 'CD42b': ['GP1BA', 'GP1BB'],
 'CD115': ['CSF1R'],
 'CD117': ['KIT'],
 'XCR1': ['XCR1'],
 'CD112': ['PVRIG', 'NECTIN2'],
 'MERTK': ['MERTK'],
 'B7-H4': ['VTCN1'],
 'CD21': ['CR2'],
 'CLEC2': ['CLEC1B'],
 'CD48': ['CD48'],
 'CD47': ['CD47'],
 'CD46': ['CD46'],
 'CD41': ['ITGA2B'],
 'CD40': ['CD40', 'CD40LG', 'TRAF3'],
 'CD43': ['SPN'],
 'CD338': ['ABCG2'],
 'CD235a': ['GYPA'],
 'CD335': ['NCR1'],
 'CD119': ['IFNGR1'],
 'CD169': ['SIGLEC1'],
 'CD28': ['CD28', 'TMIGD2'],
 'CD161': ['KLRB1'],
 'CD163': ['CD163', 'CD163L1'],
 'CD164': ['CD164', 'CD164L2'],
 'CD144': ['CDH5'],
 'CD202b': ['TEK'],
 'CD11c': ['ITGAX']}

In [2]:
PROTEIN_TO_GENES: dict[str, list[str]] = {'CD39': ['ENTPD1', 'ENTPD6', 'ENTPD2', 'ENTPD3', 'ENTPD5'],
 'CD107a': ['LAMP1'],
 'CD62P': ['SELP'],
 'CD30': ['TNFRSF8', 'TNFSF8'],
 'CD31': ['PECAM1'],
 'CD34': ['CD34'],
 'CD35': ['CR1'],
 'CD36': ['CD36', 'SCARB1', 'SCARB2'],
 'CD223': ['LAG3'],
 'TIGIT': ['TIGIT'],
 'CD226': ['CD226'],
 'CD178': ['FASLG'],
 'CD319': ['SLAMF7'],
 'CD171': ['L1CAM'],
 'Siglec-8': ['SIGLEC8'],
 'CD340': ['ERBB2'],
 'VEGFR-3': ['FLT4'],
 'CD29': ['ITGB1'],
 'CD62E': ['SELE'],
 'CD22': ['CD22'],
 'CD20': ['MS4A1', 'MS4A7', 'MS4A6A', 'MS4A4A', 'MS4A3', 'MS4A10', 'MS4A5'],
 'CD27': ['CD27', 'CD70', 'SIVA1'],
 'CD25': ['IL2RA'],
 'CD24': ['CD24'],
 'CD146': ['MCAM'],
 'Galectin-9': ['LGALS9C', 'LGALS9', 'LGALS9B'],
 'CD142': ['F3'],
 'CD141': ['THBD'],
 'CD294': ['PTGDR2'],
 'CX3CR1': ['CX3CR1'],
 'CD303': ['CLEC4C'],
 'GP130': ['LRPPRC', 'IL31RA', 'IL6ST', 'TLE5'],
 'CD253': ['TNFSF10'],
 'CD357': ['TNFRSF18'],
 'CD354': ['TREM1'],
 'CLEC12A': ['CLEC12A'],
 'Folate': ['FOLR2',
  'FOLR3',
  'FOLR1',
  'SLC19A1',
  'IZUMO1R',
  'SLC46A1',
  'FOLH1B',
  'FOLH1',
  'SLC19A4P'],
 'CD209': ['CD209', 'CLEC4M'],
 'CD152': ['CTLA4'],
 'CD154': ['CD40LG'],
 'CD155': ['PVR'],
 'Cadherin': ['CDH15',
  'CDH6',
  'CDH3',
  'CDH12',
  'CDH13',
  'CDH16',
  'CDH10',
  'CDH18',
  'CDH11',
  'CDH2'],
 'CD201': ['PROCR'],
 'CD204': ['MSR1'],
 'CD205': ['LY75'],
 'CD206': ['MRC1'],
 'CD207': ['CD207'],
 'CD1d': ['CD1D'],
 'CD284': ['TLR4'],
 'CD1c': ['CD1C'],
 'Podoplanin': ['PDPN'],
 'CD1a': ['CD1A'],
 'CD366': ['HAVCR2'],
 'IgM': ['JCHAIN', 'CD5L', 'FCMR', 'CD79A'],
 'CD49d': ['ITGA4'],
 'LOX-1': ['ALOX15', 'OLR1'],
 'TIM-4': ['TIMD4'],
 'CD98': ['SLC3A2', 'SLC7A5'],
 'CD370': ['CLEC9A'],
 'CD49a': ['ITGA1'],
 'C5L2': ['C5AR2'],
 'CD124': ['IL4R'],
 'CD127': ['IL7R'],
 'CD126': ['IL6R'],
 'CD279': ['PDCD1'],
 'CD278': ['ICOS'],
 'CD123': ['IL3RA'],
 'CD122': ['IL2RB'],
 'CD96': ['CD96'],
 'CD274': ['CD274'],
 'CD95': ['FAS', 'FASLG'],
 'CD271': ['NGFR'],
 'CD270': ['TNFRSF14'],
 'CD90': ['THY1'],
 'CD272': ['BTLA'],
 'CD16': ['FCGR3B', 'FCGR3A'],
 'CD14': ['CD14'],
 'CD13': ['ANPEP'],
 'CD267': ['TNFRSF13B'],
 'CD200': ['CD200', 'CD200R1', 'CD200R1L'],
 'CD18': ['ITGB2'],
 'CD19': ['CD19'],
 'CD194': ['CCR4'],
 'CD70': ['CD70'],
 'CD71': ['TFRC'],
 'CD72': ['CD72'],
 'CD73': ['NT5E'],
 'CD177': ['CD177'],
 'CD301': ['CLEC10A'],
 'CD140a': ['PDGFRA'],
 'CD140b': ['PDGFRB'],
 'CD305': ['LAIR1'],
 'CD304': ['NRP1'],
 'CD2': ['CD2', 'CD2BP2', 'SLAMF7', 'CD2AP', 'PSTPIP1', 'SH3KBP1', 'SLAMF9'],
 'CD309': ['KDR'],
 'CD85g': ['LILRA4'],
 'CD110': ['MPL'],
 'CD8': ['CD8B', 'CD8A', 'CD8B2'],
 'CD9': ['CD9', 'PTGFRN'],
 'HLA-DR': ['ANP32B', 'SET', 'CD74', 'ANP32A'],
 'CD137': ['TNFRSF9'],
 'CD134': ['TNFRSF4'],
 'CD135': ['FLT3'],
 'CD61': ['ITGB3'],
 'CD192': ['CCR2'],
 'CD268': ['TNFRSF13C'],
 'CD269': ['TNFRSF17'],
 'CD81': ['CD81', 'IGSF8'],
 'CD80': ['CD80'],
 'CD83': ['CD83'],
 'CD193': ['CCR3'],
 'TSLPR': ['CRLF2'],
 'CD86': ['CD86'],
 'CCR10': ['CCR10', 'ACKR2'],
 'Notch-1': ['NOTCH1'],
 'Notch-2': ['NOTCH2'],
 'CD337': ['NCR3'],
 'CD79b': ['CD79B'],
 'CD79a': ['CD79A', 'IGBP1'],
 'CD49b': ['ITGA2'],
 'CD64': ['FCGR1A'],
 'CD63': ['CD63'],
 'CD69': ['CD69'],
 'CD68': ['CD68'],
 'CD314': ['KLRK1'],
 'CD186': ['CXCR6'],
 'CD185': ['CXCR5'],
 'CD184': ['CXCR4'],
 'CD103': ['ITGAE'],
 'CD102': ['ICAM2'],
 'CD106': ['VCAM1'],
 'CD105': ['ENG'],
 'CD66b': ['CEACAM8'],
 'CD252': ['TNFSF4'],
 'CD109': ['CD109'],
 'CD158f': ['KIR2DL5B', 'KIR2DL5A'],
 'CD8a': ['CD8A'],
 'CD203c': ['ENPP3'],
 'CD52': ['CD52'],
 'CD195': ['CCR5'],
 'CD196': ['CCR6'],
 'CD54': ['ICAM1'],
 'CD55': ['CD55'],
 'CD99': ['CD99', 'CD99L2'],
 'CD59': ['CD59'],
 'CD93': ['CD93'],
 'CD244': ['CD244'],
 'CD158': ['KIR3DL3',
  'KIR2DL5B',
  'KIR2DS5',
  'KIR2DS1',
  'KIR2DL4',
  'KIR2DL2',
  'KIR3DL2',
  'KIR2DS4',
  'KIR2DS2',
  'KIR3DL1'],
 'CD273': ['PDCD1LG2'],
 'CD243': ['ABCB1'],
 'CD325': ['CDH2'],
 'CD324': ['CDH1'],
 'CD307e': ['FCRL5'],
 'CD172a': ['SIRPA'],
 'CD307d': ['FCRL4'],
 'CD42b': ['GP1BA', 'GP1BB'],
 'CD115': ['CSF1R'],
 'CD117': ['KIT'],
 'XCR1': ['XCR1'],
 'CD112': ['PVRIG', 'NECTIN2'],
 'MERTK': ['MERTK'],
 'B7-H4': ['VTCN1'],
 'CD21': ['CR2'],
 'CLEC2': ['CLEC1B'],
 'CD48': ['CD48'],
 'CD47': ['CD47'],
 'CD46': ['CD46'],
 'CD41': ['ITGA2B'],
 'CD40': ['CD40', 'CD40LG', 'TRAF3'],
 'CD43': ['SPN'],
 'CD338': ['ABCG2'],
 'CD235a': ['GYPA'],
 'CD335': ['NCR1'],
 'CD119': ['IFNGR1'],
 'CD169': ['SIGLEC1'],
 'CD28': ['CD28', 'TMIGD2'],
 'CD161': ['KLRB1'],
 'CD163': ['CD163', 'CD163L1'],
 'CD164': ['CD164', 'CD164L2'],
 'CD144': ['CDH5'],
 'CD202b': ['TEK'],
 'CD11c': ['ITGAX']}

In [3]:
PROTEIN_TO_GENES == uniprot_genes_228_adts

True